# Sentiment Analysis — OpenAI LLM Baseline

Evaluates the OpenAI backend (`gpt-5.4-mini` by default) against the same
star-rating-derived sentiment labels used in `01_baseline.ipynb` and
`02_random_forest.ipynb` — this is the model used in production
(`src/sentiment/classifier.py`, auto-selected when `OPENAI_API_KEY` is set).

In [ ]:
import sys
sys.path.append('../..')

import os
import getpass
import importlib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_recall_fscore_support
)
import matplotlib.pyplot as plt

from src.sentiment.classifier import rating_to_sentiment

df = pd.read_csv('../../datasets/clean_reviews.csv')
df['true_sentiment'] = df['star_rating'].apply(rating_to_sentiment)

## Shared test set
Same split parameters (`test_size=0.2`, `random_state=42`) as the other two
sentiment notebooks, so all three models are compared on identical data.

In [ ]:
_, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['true_sentiment']
)

LABELS = ['negative', 'neutral', 'positive']
EVAL_N = 500
eval_df = test_df.sample(n=min(EVAL_N, len(test_df)), random_state=42).reset_index(drop=True)
print(f"Evaluating on {len(eval_df)} reviews")
print(eval_df['true_sentiment'].value_counts())

## Set the API key for this session only


In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Reload so the module picks up the key fresh (its internal client is cached)
import src.sentiment.classifier as classifier_module
importlib.reload(classifier_module)

## Run predictions

In [ ]:
preds = classifier_module.predict_sentiment_batch(eval_df['review_text'].tolist())
eval_df['predicted_sentiment'] = [p['label'] for p in preds]
eval_df['confidence'] = [p['score'] for p in preds]

## Metrics

In [ ]:
acc = accuracy_score(eval_df['true_sentiment'], eval_df['predicted_sentiment'])
precision, recall, f1, _ = precision_recall_fscore_support(
    eval_df['true_sentiment'], eval_df['predicted_sentiment'], labels=LABELS, zero_division=0
)
print(f"Accuracy: {acc:.2%}  |  Macro F1: {f1.mean():.2%}\n")
print(classification_report(eval_df['true_sentiment'], eval_df['predicted_sentiment'], labels=LABELS, zero_division=0))

In [ ]:
cm = confusion_matrix(eval_df['true_sentiment'], eval_df['predicted_sentiment'], labels=LABELS)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(cmap='Greens')
plt.title('OpenAI (gpt-5.4-mini) — Confusion Matrix')
plt.tight_layout()
plt.savefig('../../datasets/confusion_matrix_openai.png')
plt.show()

## Neutral-class diagnostic
Neutral is the known-hardest class (see comparison discussion) 

In [ ]:
neutral_true = eval_df[eval_df['true_sentiment'] == 'neutral']
print(neutral_true['predicted_sentiment'].value_counts())

In [ ]:
eval_df.to_csv('../../datasets/openai_predictions.csv', index=False)

# Clear the key from this session once done
os.environ.pop("OPENAI_API_KEY", None)
print('Saved predictions to datasets/openai_predictions.csv')